# Generate Political Labels for `ukr_rus_twitter`

This notebook derives weak political labels from retweet targets and profile-description keywords, then writes a labeled graph artifact for classification experiments.

The default output is a copy of the existing graph with `y` and `label_names` populated.

In [1]:
import csv
import glob
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
CSV_GLOB = "/project2/ll_774_951/uk_ru/twitter/data/*/*.csv"


import glob
import pandas as pd
import numpy as np
import torch
from collections import defaultdict

SOURCE_GRAPH = "/home1/eibl/gfm/prodigy/midterm/graph_co_retweet/graph_data.pt"
OUTPUT_PATH  = "/home1/eibl/gfm/prodigy/midterm/graph_co_retweet/graph_data_pseudo.pt"

COLS = ['userid', 'hashtag', 'rt_hashtag', 'text', 'location', 'description', 'urls_list', 'rt_urls_list', 'date']

pd.set_option('display.max_columns', None)

In [3]:
files = sorted(glob.glob(CSV_GLOB))
print(f"Found {len(files)} CSV files")

chunks = []
skipped = 0
for f in files[:2]:
    try:
        print("reading ", f, flush=True, end='\r')
        chunks.append(pd.read_csv(f,
                                  # usecols=COLS, 
                                  engine='python', on_bad_lines='skip'))
    except Exception as e:
        print(f"Skipping {f}: {e}")
        skipped += 1
df = pd.concat(chunks, ignore_index=True)
df['userid'] = pd.to_numeric(df['userid'], errors='coerce')
df = df.dropna(subset=['userid']).copy()
df['userid'] = df['userid'].astype(int)
print(f"Skipped files: {skipped}")
print(f"Total rows: {len(df):,}  |  Unique users: {df['userid'].nunique():,}")

Found 1498 CSV files
Skipped files: 0t2/ll_774_951/uk_ru/twitter/data/2022-02/ukraine_russia-2022-02-22-05.csv
Total rows: 251,288  |  Unique users: 121,509


## URLs

In [4]:
import json
def f(x):
    try:
        return eval(x)
    except Exception as e:
        print(e)
        print(x)
        
df['urls_list'] = df.urls_list.replace({None: "[]"}).apply(f)
df['urls_list_'] = df.urls_list.explode().str['display_url'].groupby(level=0).agg(list)
df['urls_list_'] = df['urls_list_'].apply(lambda x: [y for y in x if pd.notna(y)])

df['rt_urls_list'] = df.rt_urls_list.replace({None: "[]"}).apply(f)
df['rt_urls_list_'] = df.rt_urls_list.explode().str['display_url'].groupby(level=0).agg(list)
df['rt_urls_list_'] = df['rt_urls_list_'].apply(lambda x: [y for y in x if pd.notna(y)])

df['qtd_urls_list'] = df.qtd_urls_list.replace({None: "[]"}).apply(f)
df['qtd_urls_list_'] = df.qtd_urls_list.explode().str['display_url'].groupby(level=0).agg(list)
df['qtd_urls_list_'] = df['qtd_urls_list_'].apply(lambda x: [y for y in x if pd.notna(y)])

In [13]:
df['tweet_url'] = 'www.x.com/' + df.screen_name + '/status/' + df.tweetid

In [38]:
outlet_scores = {
'abcnews.go.com': '2',
'bbc.com': '3',
'breitbart.com': '5',
'bostonglobe.com': '2',
'businessinsider.com': '3',
'buzzfeednews.com': '1',
'cbsnews.com': '2',
'chicagotribune.com': '3',
'cnbc.com': '3',
'cnn.com': '2',
'dailycaller.com': '5',
'dailymail.co.uk': '5',
'foxnews.com': '4',
'huffpost.com': '1',
'infowars.com': '5',
'latimes.com': '2',
'msnbc.om': '1',
'nbcnews.com': '2',
'nytimes.com': '2',
'npr.org': '3',
'oann.com': '4',
'pbs.org': '3',
'reuters.com': '3',
'theguardian.com': '2',
'usatoday.com': '3',
'yahoo.com': '2',
'vice.com': '1',
'washingtonpost.com': '2',
'wsj.com': '3',
}

additional = {
    'thehill.com': -0.8,
    'rt.com': 2,
    'rawstory.com': -4,
    'news.sky.com': -2,
    'independent.co.uk': -2,
    'dailykos.com': -4
}

def map_value(value, in_min=-6, in_max=6, out_min=1, out_max=5):
    return out_min + (value - in_min) * (out_max - out_min) / (in_max - in_min)

additional = {k: map_value(v) for k, v in additional.items()}
outlet_scores = outlet_scores | additional

In [39]:
domain_to_canonical = {
    
    'bit.ly': None,  # URL shortener
    'dlvr.it': None,  # syndication tool
    'trib.al': None,  # URL shortener
    'ift.tt': None,  # IFTTT shortener
    'twibbon.com': None,  # campaign tool
    't.me': None,  # Telegram shortener
    'apple.news': None,  # aggregator
    'ow.ly': None,  # Hootsuite shortener
    'buff.ly': None,  # Buffer shortener
    'tinyurl.com': None,  # URL shortener
    'news.google.com': None,  # aggregator
    'lnr.app': None,  # URL shortener
    'fiverr.com': None,  # freelancing platform, likely data artifact
    
    'twitter.com': 'twitter.com',
    'reut.rs': 'reuters.com',
    'youtu.be': 'youtube.com',
    'theguardian.com': 'theguardian.com',
    'youtube.com': 'youtube.com',
    'nytimes.com': 'nytimes.com',
    'reuters.com': 'reuters.com',
    'mfa.gov.tr': 'mfa.gov.tr',
    'washingtonpost.com': 'washingtonpost.com',
    'cnn.com': 'cnn.com',
    'businessinsider.com': 'businessinsider.com',
    'rt.com': 'rt.com',
    'hill.cm': 'thehill.com',
    'bbc.in': 'bbc.com',
    'foxnews.com': 'foxnews.com',
    'facebook.com': 'facebook.com',
    'rawstory.com': 'rawstory.com',
    'bbc.com': 'bbc.com',
    'bbc.co.uk': 'bbc.com',
    'news.sky.com': 'news.sky.com',
    'melenchon.fr': 'melenchon.fr',
    'a.msn.com': 'msn.com',
    'npr.org': 'npr.org',
    'politico.com': 'politico.com',
    'en.wikipedia.org': 'wikipedia.org',
    'cnb.cx': 'cnbc.com',
    'timesofindia.indiatimes.com': 'timesofindia.indiatimes.com',
    'dailykos.com': 'dailykos.com',
    'independent.co.uk': 'independent.co.uk',
    'bfmtv.com': 'bfmtv.com',
    'msn.com': 'msn.com',
    'jp.reuters.com': 'reuters.com',
    'axios.com': 'axios.com',
    'babylonbee.com': 'babylonbee.com',
    'news.yahoo.com': 'yahoo.com',
    'gelecektenhaber.com': 'gelecektenhaber.com',
    'instagram.com': 'instagram.com',
    'google.com': 'google.com',
    'zerohedge.com': 'zerohedge.com',
    'tagesschau.de': 'tagesschau.de',
    'thehill.com': 'thehill.com',
    'cnbc.com': 'cnbc.com',
    'bloomberg.com': 'bloomberg.com',
    'lemonde.fr': 'lemonde.fr',
    'ndtv.com': 'ndtv.com',
    'forbes.com': 'forbes.com',
    'ti.me': 'time.com',
    'aninews.in': 'aninews.in',
    'wsj.com': 'wsj.com',
    'theatlantic.com': 'theatlantic.com',
    'spiegel.de': 'spiegel.de',
    'on.ft.com': 'ft.com',
    'en.kremlin.ru': 'kremlin.ru',
    'cnn.it': 'cnn.com',
    'whitehouse.gov': 'whitehouse.gov',
    'insiderpaper.com': 'insiderpaper.com',
    'abc.net.au': 'abc.net.au',
    'francetvinfo.fr': 'francetvinfo.fr',
    'edition.cnn.com': 'cnn.com',
    'on.rt.com': 'rt.com',
    'aljazeera.com': 'aljazeera.com',
    'mobile.twitter.com': 'twitter.com',
    'parafesor.net': 'parafesor.net',
    'who.int': 'who.int',
    'world.bistosh.com': 'bistosh.com',
    'es.rt.com': 'rt.com',
    'coinkurry.com': 'coinkurry.com',
    'presidentti.fi': 'presidentti.fi',
    'nos.nl': 'nos.nl',
    'sputniknews.com': 'sputniknews.com',
    'mol.im': 'dailymail.co.uk',
    'vilaweb.cat': 'vilaweb.cat',
    'bild.de': 'bild.de',
    'washingtontimes.com': 'washingtontimes.com',
    'scmp.com': 'scmp.com',
    'wapo.st': 'washingtonpost.com',
    'aajtak.in': 'aajtak.in',
    'nbcnews.com': 'nbcnews.com',
    'tf1info.fr': 'tf1info.fr',
    'rumble.com': 'rumble.com',
    'maisinwin.blogspot.com': 'maisinwin.blogspot.com',
    'welt.de': 'welt.de',
    'nypost.com': 'nypost.com',
    'ft.com': 'ft.com',
    'liveuamap.com': 'liveuamap.com',
    'moneycontrol.com': 'moneycontrol.com',
    'kremlin.ru': 'kremlin.ru',
    
}

In [47]:
# udf = urls.to_frame()
# udf[''] = 
result = (
    df.urls_list_.explode()
    .str.split('/').str[0]
    .map(domain_to_canonical)
    .map(outlet_scores)
    .astype(float)
    # .dropna()
    .groupby(level=0)
    .agg(list)
)
# result['scores']

In [50]:
df['url_scores'] = result
df['url_scores'] = df.url_scores.apply(lambda y: [x for x in y if pd.notna(x)])

In [71]:
df['mean_tweet_url_pol_score'] = df.url_scores.apply(np.mean, 1)

/tmp/SLURM_7786922/ipykernel_777750/340154879.py:1: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  df['mean_tweet_url_pol_score'] = df.url_scores.apply(np.mean, 1)
/home1/eibl/.conda/envs/prodigy/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home1/eibl/.conda/envs/prodigy/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [61]:
handle2score = {
    "ABC": 2,
    "BBCWorld": 3,
    "BreitbartNews": 5,
    "BostonGlobe": 2,
    "businessinsider": 3,
    "BuzzFeedNews": 1,
    "CBSNews": 2,
    "chicagotribune": 3,
    "CNBC": 3,
    "CNN": 2,
    "DailyCaller": 5,
    "DailyMail": 5,
    "FoxNews": 4,
    "HuffPost": 1,
    "InfoWars*": 5,
    "latimes": 2,
    "MSNBC": 1,
    "NBCNews": 2,
    "nytimes": 2,
    "NPR": 3,
    "OANN": 4,
    "PBS": 3,
    "Reuters": 3,
    "guardian": 2,
    "USATODAY": 3,
    "YahooNews": 2,
    "VICE": 1,
    "washingtonpost": 2,
    "WSJ": 3,
    
    # additional, using the ratings caluclated earlier
    "thehill": 2.7333333333333334,
    "RT_com": 3.6666666666666665,
    "RawStory": 1.6666666666666665,
    "SkyNews": 2.333333333333333,
    "Independent": 2.333333333333333,
    "dailykos": 1.6666666666666665
}

handle2score = {k.lower(): v for k,v in handle2score.items()}

In [80]:
df['rt_org_pol_score'] = df.rt_screen.str.lower().map(handle2score)

In [85]:
df['tweet_url_pol_score_is_left'] = df.mean_tweet_url_pol_score.map(lambda x: True if x < 3 else False if x > 3 else np.nan)
df['rt_org_pol_score_is_left'] = df.rt_org_pol_score.map(lambda x: True if x < 3 else False if x > 3 else np.nan)

In [111]:
df['mean_tweet_pol_score'] = df[['mean_tweet_url_pol_score', 'rt_org_pol_score']].mean(1)

In [132]:
df['is_left'] = df.mean_tweet_pol_score.mean()
df['is_right'] = df.mean_tweet_pol_score.gt(3)

In [151]:
user2pol = df.groupby('userid').mean_tweet_pol_score.mean()

In [164]:
user2pol = user2pol.apply(lambda x: True if x < 3 else False if x > 3 else np.nan)

In [169]:
user2pol.notna().mean()

0.020994329638133802

In [174]:
user2pol

userid
224                    NaN
645                    NaN
785                    NaN
1378                   NaN
1541                   NaN
                      ... 
1495997093239918336    NaN
1495997945375830016    NaN
1495998556179603456    NaN
1495998972397342720    NaN
1495999426892025856    NaN
Name: mean_tweet_pol_score, Length: 121509, dtype: object